In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Charger le même fichier que pour la pluie
df = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

# Convertir les dates
df["date"] = pd.to_datetime(
    df["Date/heure locale (UTC+2)"]
)

# Les quatre stations d'oxygène
stations = ["Q01", "Q02", "Q03", "Q04"]

# Afficher leurs concentrations
print(df[stations].describe())

# Dessiner les quatre courbes
plt.figure(figsize=(12, 5))

for station in stations:
    plt.plot(df["date"], df[station], label=station)

plt.title("Oxygène dissous dans la Senne — 24 juillet 2025")
plt.xlabel("Heure")
plt.ylabel("Oxygène dissous (mg/L)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Préparer les données
Dans le code si en bas on as 10.0 : c'est une référence provisoire de normalisation commune aux quatre stations. Elle sera gardé identique entre stations et entre événements, sauf justification explicite d'un changement.

In [ ]:
import numpy as np

stations = ["Q01", "Q02", "Q03", "Q04"]

# Même chronologie pour les quatre stations
temps = np.linspace(0, 1, len(df))

# Échelle commune : 0 à 10 mg/L
reference_oxygene = 10.0

# Préparer une série normalisée par station
oxygene_normalise = {}

for station in stations:
    valeurs = df[station].to_numpy(dtype=float)

    oxygene_normalise[station] = np.clip(
        valeurs / reference_oxygene,
        0,
        1
    )

Sonifier les quatre stations avec STRAUSS

In [ ]:
from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer

duree = 60
sons_oxygene = {}

positions = {
    "Q01": 0.25,
    "Q02": 0.42,
    "Q03": 0.58,
    "Q04": 0.75
}

for station in stations:

    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": oxygene_normalise[station],
        "volume": 0.35,
        "azimuth": positions[station],
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(
        score,
        sources,
        generator,
        "stereo"
    )

    soni.render()

    sons_oxygene[station] = soni

    print("Sonification terminée :", station)

Écouter les stations individuellement

In [ ]:
# Écouter Q01
sons_oxygene["Q01"].notebook_display()
# Écouter Q02
sons_oxygene["Q02"].notebook_display()
# Écouter Q03
sons_oxygene["Q03"].notebook_display()
# Écouter Q04
sons_oxygene["Q04"].notebook_display()

Pour les écouter toutes ensmeble

In [ ]:
import numpy as np
from IPython.display import Audio, display

pistes = []
frequence = None

for station in stations:
    sorties = sons_oxygene[station].out_channels

    # Récupérer les échantillons des canaux gauche et droit
    canaux = []

    for stream in sorties.values():
        audio = np.asarray(stream.values, dtype=float).ravel()
        canaux.append(audio)

        if frequence is None:
            frequence = int(stream.samprate)
        elif int(stream.samprate) != frequence:
            raise ValueError("Fréquences différentes")

    piste = np.stack(canaux)
    pistes.append(piste)

# Mélanger les quatre stations
mix = np.mean(pistes, axis=0)

# Ajuster le volume pour éviter la saturation
niveau_max = np.max(np.abs(mix))

if niveau_max > 0:
    mix = 0.8 * mix / niveau_max

# Écouter le mélange
display(Audio(mix, rate=frequence, normalize=False))

Á verifier apres ecoute: 
1. Entends-tu plusieurs sons distincts ou un seul son difficile à comprendre ?
2. Peux-tu percevoir les fortes baisses d'oxygène en soirée ?
3. La spatialisation gauche-droite aide-t-elle réellement à distinguer les stations ?

In [ ]:
stream = sons_oxygene["Q01"].out_channels

print(type(stream))
print(type(next(iter(stream.values()))))
print(dir(next(iter(stream.values()))))

Exporter l'oxygene en wav

In [ ]:
import numpy as np
from scipy.io import wavfile
from pathlib import Path

# Créer le dossier de sortie
dossier_sortie = Path("../audio")
dossier_sortie.mkdir(parents=True, exist_ok=True)

fichier_audio = dossier_sortie / "senne_oxygene.wav"

# Vérifier la durée actuelle
duree_audio = mix.shape[1] / frequence

print("Durée actuelle :", duree_audio, "secondes")

# Adapter le son à 60 secondes
nombre_echantillons = int(frequence * 60)

audio_final = mix[:, :nombre_echantillons]

if audio_final.shape[1] < nombre_echantillons:
    audio_final = np.pad(
        audio_final,
        (
            (0, 0),
            (0, nombre_echantillons - audio_final.shape[1])
        )
    )

# Convertir en WAV 16 bits
audio_final = np.nan_to_num(audio_final)
audio_wav = (
    np.clip(audio_final.T, -1, 1) * 32767
).astype(np.int16)

# Enregistrer
wavfile.write(
    str(fichier_audio),
    int(frequence),
    audio_wav
)

print("WAV enregistré :", fichier_audio)
print("Durée finale :", len(audio_wav) / frequence, "secondes")

In [ ]:
from IPython.display import Audio, display

display(Audio(filename=str(fichier_audio)))

Fusionner le son et la video

In [ ]:
import subprocess
import imageio_ffmpeg
from pathlib import Path

# Localisation des fichiers
video = Path("../outputs/senne_oxygene_sans_son.mp4")
audio = Path("../audio/senne_oxygene.wav")

# Vidéo finale
sortie = Path("../outputs/senne_oxygene_avec_son.mp4")

# Vérifier que les fichiers existent
print("Vidéo trouvée :", video.exists())
print("Audio trouvé :", audio.exists())

if not video.exists() or not audio.exists():
    raise FileNotFoundError(
        "Vérifie les chemins des fichiers audio et vidéo."
    )

# Récupérer FFmpeg
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

# Fusionner sans réencoder la vidéo
commande = [
    ffmpeg,
    "-y",
    "-i", str(video),
    "-i", str(audio),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-b:a", "192k",
    "-t", "60",
    str(sortie)
]

subprocess.run(commande, check=True)

print("Vidéo avec son enregistrée :", sortie)

Regarder et écouter le résultat

In [ ]:
from IPython.display import Video, display

display(
    Video(
        filename=str(sortie),
        embed=True
    )
)